# Precompute 01 — embed the candidate pool on a Colab GPU

Runs the **same** `src/precompute/01_embed_candidates.py` as locally, but on the T4,
so the full 100k pool takes ~10 min instead of ~14 h on CPU.

**This is offline precompute — GPU + network are allowed here.** `rank.py` never does
any of this; it only loads the artifacts produced. Output = three files
(`candidate_embeddings.npy`, `candidate_ids.npy`, `embeddings_meta.json`).

### The data path (read this once)
Your T4 kernel runs on a **remote Colab VM** — it cannot see your laptop's disk. So the
data must reach the VM and the artifacts must come back. **Google Drive is the channel**
(you're already Google-authenticated via Colab).

**Before running:** upload **one** of these to your Drive root (`MyDrive/`):
* `candidates_compact.jsonl.gz`  ← **recommended, ~9 MB**, identical embedding text
  (staged locally at `redrob-ranker/data/candidates_compact.jsonl.gz`), or
* the full `candidates.jsonl` (487 MB).

Also set **Runtime → Change runtime type → GPU** if not already.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi

## 2. Install sentence-transformers
Use Colab's preinstalled **CUDA** torch — do **not** install our `requirements.txt`
(it pins the CPU-only torch for the `rank.py` path).

In [ ]:
!pip install -q sentence-transformers

## 3. Get the code (clone the repo onto the VM)

In [ ]:
import os

REPO_URL = "https://github.com/CoffeeAurCode/redrob-ranker.git"
if not os.path.isdir("redrob-ranker"):
    !git clone $REPO_URL
%cd redrob-ranker

## 4. Mount Drive and stage the data
Mounting prompts for a one-time authorization. This auto-detects whichever file you
uploaded (compact `.gz` preferred), unzips if needed, and places it where the script
expects. `wc -l` must print **100000**.

In [ ]:
import shutil, subprocess
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive"

candidates = [
    f"{DRIVE}/candidates_compact.jsonl.gz",  # recommended (~9 MB)
    f"{DRIVE}/candidates.jsonl.gz",
    f"{DRIVE}/candidates.jsonl",             # full pool
]
src = next((p for p in candidates if os.path.exists(p)), None)
assert src, f"Upload candidates_compact.jsonl.gz (or candidates.jsonl) to {DRIVE}"

os.makedirs("data", exist_ok=True)
dst = "data/candidates.jsonl"
if src.endswith(".gz"):
    subprocess.run(f'gunzip -c "{src}" > {dst}', shell=True, check=True)
else:
    shutil.copy(src, dst)
print("staged from:", src)
!wc -l {dst}

## 5. Embed the pool (GPU auto-detected)
`SentenceTransformer` picks CUDA automatically. Checkpoints under
`artifacts/.embed_ckpt/`, so if the session drops, re-run this cell to resume.

In [ ]:
!python src/precompute/01_embed_candidates.py

## 6. Sanity check
Top 30 by similarity to a hand-written "ideal Senior AI Engineer" query. You want
ML / retrieval / ranking / recsys / search / data-science titles near the top — **not**
a keyword-stuffed `Marketing Manager` (that would mean skills are leaking, which they
should not be).

In [ ]:
!python src/precompute/sanity_rank.py --top 30

## 7. Artifacts in both places: repo `artifacts/` **and** Drive
The embedding script already wrote the three files into the repo's `artifacts/` on the
VM (that is the canonical copy you commit). This cell also copies them to a **separate
Drive folder** `MyDrive/redrob_artifacts/` so you can pull them down. On your machine:
download the Drive copy into your local repo's `artifacts/`, run `pytest -q`, and commit.

In [ ]:
OUT = f"{DRIVE}/redrob_artifacts"
os.makedirs(OUT, exist_ok=True)
for name in ("candidate_embeddings.npy", "candidate_ids.npy", "embeddings_meta.json"):
    shutil.copy(f"artifacts/{name}", f"{OUT}/{name}")

print("== repo artifacts/ (canonical copy on the VM clone) ==")
!ls -lh artifacts/*.npy artifacts/*.json
print("\n== Drive copy:", OUT, "==")
!ls -lh "$OUT"
print("\n--- embeddings_meta.json ---")
!cat artifacts/embeddings_meta.json